<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TWO_world_AGENTIC_GGUF_G4_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/frank-morales2020/AST/blob/main/Qwen3dot8_27B_topo_cbp_gguf.ipynb



https://github.com/frank-morales2020/AST/blob/main/APAC_AGENTIC_GGUF.ipynb

## setup

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
!ls -ltha /root/.config/Google/DriveFS/107621623640896046014/

In [ ]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/*.gguf

In [ ]:
!apt-get update && !apt-get install -y git
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --force-reinstall --no-cache-dir git+https://github.com/abetlen/llama-cpp-python.git --verbose

In [ ]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


GPU Offload Supported: True


In [ ]:
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G  576M  236G   1% /


In [ ]:
import os
import struct
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/GGUF_Models")

def read_gguf_header(filepath):
    try:
        with open(filepath, "rb") as f:
            magic = f.read(4)
            if magic != b"GGUF":
                return {"error": "Invalid magic bytes"}
            version = struct.unpack("<I", f.read(4))[0]
            tensor_count = struct.unpack("<Q", f.read(8))[0]
            kv_count = struct.unpack("<Q", f.read(8))[0]
            return {
                "version": version,
                "tensor_count": tensor_count,
                "kv_count": kv_count
            }
    except Exception as e:
        return {"error": str(e)}

print(f"{'Model Filename':<52} | {'Size (GB)':<10} | {'GGUF Version':<12} | {'Status'}")
print("-" * 95)

total_size = 0
for file_path in sorted(model_dir.glob("*.gguf")):
    file_size = file_path.stat().st_size
    size_gb = file_size / (1024**3)
    total_size += file_size

    meta = read_gguf_header(file_path)
    if "error" not in meta:
        status = "Verified"
        version_str = f"v{meta['version']}"
    else:
        status = meta["error"]
        version_str = "N/A"

    print(f"{file_path.name:<52} | {size_gb:>8.2f}GB | {version_str:<12} | {status}")

print("-" * 95)
print(f"Total Portfolio Storage: {total_size / (1024**3):.2f} GB across {len(list(model_dir.glob('*.gguf')))} models.")

Model Filename                                       | Size (GB)  | GGUF Version | Status
-----------------------------------------------------------------------------------------------
certified-scorpion-topo-qwen-bf16.gguf               |    14.19GB | v3           | Verified
deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf |    69.83GB | v3           | Verified
deepseek_moe_16b_topocbp_q4_k_m.gguf                 |    10.11GB | v3           | Verified
gpt-oss-20b-topo-governed-Q4_K_M.gguf                |    14.72GB | v3           | Verified
llama-3.1-70b-topo-governed-q4_k_m.gguf              |    39.60GB | v3           | Verified
mixtral_8x22b_q4_k_m.gguf                            |    79.71GB | v3           | Verified
muse-glimmer-30b-card-faithful-Q4_K_M.gguf           |    15.77GB | v3           | Verified
nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf          |     4.97GB | v3           | Verified
qwen3.8-27b-topocbp-Q8_0.gguf                        |    26.63GB | v3       

## TWO MODELS AGENTIC

In [ ]:
import os
import re
import shutil
import time
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama


def clean_output(text: str, max_repeat_lines: int = 2) -> str:
    cleaned = re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL)
    if "<think>" in cleaned:
        cleaned = cleaned.split("<think>", 1)[0]
    cleaned = cleaned.replace("</think>", "").strip()

    lines = cleaned.splitlines()
    deduped = []
    prev_norm = None
    streak = 0
    for line in lines:
        norm = re.sub(r"\s+", " ", line.strip().lstrip("-*•0123456789. ")).lower()
        if norm and norm == prev_norm:
            streak += 1
            if streak >= max_repeat_lines:
                continue
        else:
            prev_norm = norm
            streak = 1
        deduped.append(line)
    return "\n".join(deduped).strip()


def get_mem_available_gb() -> float | None:
    try:
        with open("/proc/meminfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("MemAvailable:"):
                    kb = int(line.split()[1])
                    return kb / (1024**2)
    except OSError:
        pass
    return None


class TwoModelAgent:
    MODELS = {
        "apac_router": {
            "label": "APAC Router (DeepSeek MoE 16B)",
            "repo_id": "frankmorales2020/deepseek-moe-16b-topo-cbp-merged-gguf",
            "filename": "deepseek_moe_16b_topocbp_q4_k_m.gguf",
            "n_ctx": 4096,
            "n_gpu_layers": -1,
            "flash_attn": True,
        },
        "validator": {
            "label": "Foundational Validator (Certified Scorpion)",
            "repo_id": "frankmorales2020/certified-scorpion-topo-cbp-qwen-master-GGUF",
            "filename": "certified-scorpion-topo-qwen-bf16.gguf",
            "n_ctx": 4096,
            "n_gpu_layers": -1,
            "flash_attn": True,
        },
    }

    def __init__(
        self,
        local_dir="/content/hf_gguf_models",
        hf_token=None,
        delete_after_load=True,
        min_free_ram_gb=32.0,
        verbose_on_error=True,
    ):
        self.local_dir = Path(local_dir)
        self.local_dir.mkdir(parents=True, exist_ok=True)
        self.hf_token = hf_token or os.environ.get("HF_TOKEN")
        self.delete_after_load = delete_after_load
        self.min_free_ram_gb = min_free_ram_gb
        self.verbose_on_error = verbose_on_error
        self.api = HfApi(token=self.hf_token)
        self.models: dict[str, Llama | None] = {}

        for key in self.MODELS:
            avail_ram = get_mem_available_gb()
            if avail_ram is not None and avail_ram < self.min_free_ram_gb:
                print(
                    f"[RAM GUARD] Skipping {key}: MemAvailable is {avail_ram:.1f} GB "
                    f"(< {self.min_free_ram_gb:.1f} GB floor)."
                )
                self.models[key] = None
                setattr(self, key, None)
                continue
            llm = self._load_model(key)
            setattr(self, key, llm)

        active = [k for k, m in self.models.items() if m is not None]
        avail_end = get_mem_available_gb()
        ram_msg = f" | MemAvailable={avail_end:.1f} GB" if avail_end is not None else ""
        print(f"Loaded {len(active)}/2 models: {active}{ram_msg}")

    def _cleanup_disk_artifacts(self, path: Path):
        if not self.delete_after_load:
            return
        if path.exists():
            try:
                size_gb = path.stat().st_size / (1024**3)
                path.unlink()
                print(f"Deleted {path.name} immediately after load (freed ~{size_gb:.1f} GB)")
            except OSError as exc:
                print(f"[WARN] Could not delete {path.name}: {exc}")
        for cache_path in [self.local_dir / ".cache", Path.home() / ".cache" / "huggingface"]:
            if cache_path.exists():
                try:
                    shutil.rmtree(cache_path)
                except OSError:
                    pass

    def _resolve_repo_filename(self, repo_id: str, preferred_filename: str) -> str:
        try:
            files = self.api.list_repo_files(repo_id=repo_id)
            ggufs = [f for f in files if f.lower().endswith(".gguf")]
            if preferred_filename in ggufs:
                return preferred_filename
            if ggufs:
                return ggufs[0]
        except Exception:
            pass
        return preferred_filename

    def _load_model(self, key: str) -> Llama | None:
        spec = self.MODELS[key]
        filename = self._resolve_repo_filename(spec["repo_id"], spec["filename"])
        local_target = self.local_dir / Path(filename).name

        if not (local_target.exists() and local_target.stat().st_size > 0):
            for attempt in range(1, 4):
                t0 = time.time()
                print(f"Downloading {spec['label']} [{filename}] (attempt {attempt}/3)...")
                try:
                    downloaded = hf_hub_download(
                        repo_id=spec["repo_id"],
                        filename=filename,
                        local_dir=str(self.local_dir),
                        token=self.hf_token,
                    )
                    local_target = Path(downloaded)
                    print(f"Downloaded {local_target.name} in {time.time() - t0:.1f}s")
                    break
                except Exception as exc:
                    print(f"[WARN] Download attempt {attempt}/3 failed for {spec['label']}: {exc}")
                    if local_target.exists():
                        try:
                            local_target.unlink()
                        except OSError:
                            pass
                    if attempt < 3:
                        time.sleep(10 * attempt)
            else:
                print(f"[HF DOWNLOAD FAILED] {spec['label']}")
                self.models[key] = None
                return None

        configs = [
            {
                "n_ctx": spec["n_ctx"],
                "n_gpu_layers": spec["n_gpu_layers"],
                "flash_attn": spec["flash_attn"],
                "use_mmap": False,
                "verbose": False,
            },
            {
                "n_ctx": 4096,
                "n_gpu_layers": spec["n_gpu_layers"],
                "flash_attn": False,
                "use_mmap": False,
                "verbose": self.verbose_on_error,
            },
        ]
        last_err = None
        for idx, cfg in enumerate(configs, start=1):
            try:
                llm = Llama(model_path=str(local_target), **cfg)
                print(f"Loaded {spec['label']} (profile {idx}, n_gpu_layers={spec['n_gpu_layers']})")
                self.models[key] = llm
                self._cleanup_disk_artifacts(local_target)
                return llm
            except Exception as exc:
                last_err = exc
        print(f"[FAILED] {spec['label']}: {last_err}")
        self.models[key] = None
        return None

    def _run(
        self,
        key: str,
        prompt: str,
        max_tokens: int = 1024,
        temperature: float = 0.3,
        repeat_penalty: float = 1.15,
        stop=None,
    ) -> str:
        llm = self.models.get(key)
        if llm is None:
            llm = self._load_model(key)
        if llm is None:
            return f"[UNAVAILABLE: {key}]"
        res = llm(
            prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            repeat_penalty=repeat_penalty,
            stop=stop,
        )
        return clean_output(res["choices"][0]["text"] or "")

    def solve(self, query: str) -> str:
        framing = self._run(
            "apac_router",
            (
                "<|im_start|>user\n"
                f"List 5 concise, distinct structural and quantitative risk dimensions for:\n{query}\n"
                "<|im_end|>\n<|im_start|>assistant\n"
            ),
            max_tokens=384,
            temperature=0.2,
            repeat_penalty=1.15,
            stop=["<|im_end|>"],
        )
        return self._run(
            "validator",
            (
                "<|im_start|>system\nBe concise, non-repetitive, and analytical.<|im_end|>\n"
                "<|im_start|>user\n"
                "Write a concise response with distinct points under these headings: "
                "Objective, Quantitative Model, Governance Constraints, Tail Risks, and Findings.\n"
                "Do not repeat sentences or bullet points.\n\n"
                f"Query: {query}\n\nKey Dimensions:\n{framing}"
                "<|im_end|>\n<|im_start|>assistant\n"
            ),
            max_tokens=1024,
            temperature=0.35,
            repeat_penalty=1.15,
            stop=["<|im_end|>"],
        )


if __name__ == "__main__":
    agent = TwoModelAgent(
        local_dir="/content/hf_gguf_models",
        delete_after_load=True,
        min_free_ram_gb=32.0,
    )
    query = "Model the impact of cross-border liquidity shocks on multi-currency portfolio Value-at-Risk."
    print(agent.solve(query))

deepseek_moe_16b_topocbp_q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 10.9GB            

deepseek_moe_16b_topocbp_q4_k_m.gguf: downloading bytes:           |  0.00B            

Downloaded deepseek_moe_16b_topocbp_q4_k_m.gguf in 302.6s
Loaded APAC Router (DeepSeek MoE 16B) (profile 1, n_gpu_layers=-1)
Deleted deepseek_moe_16b_topocbp_q4_k_m.gguf immediately after load (freed ~10.1 GB)


certified-scorpion-topo-qwen-bf16.gguf: reconstructing file:   0%|          |  0.00B / 15.2GB            

certified-scorpion-topo-qwen-bf16.gguf: downloading bytes:           |  0.00B            

Downloaded certified-scorpion-topo-qwen-bf16.gguf in 36.9s
Loaded Foundational Validator (Certified Scorpion) (profile 1, n_gpu_layers=-1)
Deleted certified-scorpion-topo-qwen-bf16.gguf immediately after load (freed ~14.2 GB)
Loaded 2/2 models: ['apac_router', 'validator'] | MemAvailable=169.7 GB
I will provide a concise response with distinct points under the headings: Objective, Quantitative Model, Governance Constraints, Tail Risks, and Findings.

Objective:
The objective is to model the impact of cross-border liquidity shocks on multi-currency portfolio Value-at-Risk (VaR) in order to assess potential losses due to sudden changes in currency exchange rates or market disruptions. This analysis will help identify vulnerabilities within a portfolio's exposure to different currencies and inform risk management strategies.

Quantitative Model:
To achieve this objective, we propose using the Monte Carlo simulation method combined with historical data analysis for VaR modeling of multi-cu

In [ ]:
!nvidia-smi

Fri Nov 20 05:24:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   36C    P0             87W /  600W |   26115MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import re
import numpy as np


def clean_output(text: str) -> str:
    cleaned = re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL)
    if "<think>" in cleaned:
        cleaned = cleaned.split("<think>", 1)[0]
    cleaned = cleaned.replace("</think>", "").replace("<|im_end|>", "").strip()

    for marker in ["\nScenario:", "\nMetrics:", "\nQuery:", "\nWrite a concise", "\n<|im_start|>"]:
        idx = cleaned.find(marker)
        if idx != -1:
            cleaned = cleaned[:idx].strip()

    blocks = [b.strip() for b in re.split(r"\n\s*\n", cleaned) if b.strip()]
    seen, kept = set(), []
    for block in blocks:
        norm = re.sub(r"\s+", " ", block).lower()
        if norm not in seen:
            seen.add(norm)
            kept.append(block)
    return "\n\n".join(kept).strip()


def solve_4d_stress(agent, query: str, metrics: dict) -> str:
    framing_prompt = (
        f"Scenario: {query}\n"
        f"Metrics: {metrics}\n\n"
        "List 5 concise technical bullet points covering 4D rerouting, sector congestion, "
        "convective weather, flow restrictions, and fuel margins:\n"
        "- "
    )
    framing_raw = agent._run(
        "apac_router",
        framing_prompt,
        max_tokens=260,
        temperature=0.2,
        repeat_penalty=1.18,
        stop=["\nScenario:", "\nMetrics:", "\n###", "<|im_end|>"],
    )
    framing = clean_output("- " + framing_raw.lstrip("- "))

    validator_prompt = (
        f"Scenario: {query}\n"
        f"Metrics: {metrics}\n"
        f"Key Points:\n{framing}\n\n"
        "Using ONLY the exact numbers in Metrics above, write concise bullet points under "
        "Objective, Quantitative Model, Governance Constraints, Tail Risks, and Findings.\n\n"
        "Objective:\n- "
    )
    analysis_raw = agent._run(
        "validator",
        validator_prompt,
        max_tokens=650,
        temperature=0.2,
        repeat_penalty=1.18,
        stop=["\nScenario:", "\nMetrics:", "\nWrite a concise", "<|im_end|>"],
    )
    return clean_output("Objective:\n- " + analysis_raw.lstrip("- "))


result = solve_4d_stress(agent, query, metrics)
print("Metrics:", metrics)
print("\nFinal Analysis:\n", result)

Metrics: {'flights_simulated': 48, 'rerouted_flights': 30, 'total_delay_min': 444.6, 'avg_delay_min_per_flight': 9.26, 'avg_delay_min_per_rerouted_flight': 14.82, 'peak_sector_utilization': 0.6, 'low_fuel_margin_flights': 0}

Final Analysis:
 Objective:
- 4D rerouting: Simulate real-time 30 aircraft trajectories and sector congestion propagation when a major convective weather cell hits simultaneously across North Atlantic corridor.
- Sector Congestion: Model real time 30 aircraft trajectories and sector congestion propagation when a regional air traffic flow restriction hits simultaneously across Northeast corridor.

Quantitative Model:
- Flights simulated: 48
- Rerouted flights: 30 (62.5% of total)
- Total delay in minutes: 444.6 min

Governance Constraints:
- Avg delay per flight: 9.26 min
- Avg delay per rerouted flight: 14.82 min
- Peak sector utilization: 0.6 (low)

Tail Risks:
- Low fuel margin flights: None detected in simulation.

Findings:
- The model successfully simulated r